In [1]:
import os
os.chdir("..")

In [ ]:
#Extract DSS file paths from either a directory or a STAC catalog
from __future__ import annotations

from pathlib import Path
from typing import List

import pystac


def collect_dss_file_paths(path: str | Path) -> List[str]:
    """
    Collect DSS file paths from either a directory or a STAC catalog.

    This function determines whether the provided path is:
        1. A directory containing `.dss` files (searched recursively), or
        2. A STAC catalog JSON file containing `.dss` assets.

    Parameters
    ----------
    path : str | Path
        Path to a folder containing `.dss` files or a STAC `catalog.json`.

    Returns
    -------
    List[str]
        A list of absolute paths to discovered `.dss` files.

    Raises
    ------
    FileNotFoundError
        If the provided path does not exist.
    ValueError
        If no `.dss` files are found or if the path is not a valid
        directory or STAC catalog.
    """
    input_path = Path(path).resolve()

    if not input_path.exists():
        msg = f"Path does not exist: {input_path}"
        raise FileNotFoundError(msg)

    dss_files: list[str] = []

    # Case 1: Directory containing DSS files
    if input_path.is_dir():
        dss_files = [
            str(file.resolve())
            for file in input_path.rglob("*.dss")
            if file.is_file()
        ]

    # Case 2: STAC catalog JSON
    elif input_path.is_file() and input_path.suffix.lower() == ".json":
        try:
            catalog = pystac.Catalog.from_file(str(input_path))
        except Exception as exc:  # noqa: BLE001
            msg = f"File is not a valid STAC catalog: {input_path}"
            raise ValueError(msg) from exc

        for item in catalog.get_all_items():
            for asset in item.assets.values():
                href = asset.href
                if href and href.lower().endswith(".dss"):
                    absolute_href = asset.get_absolute_href()
                    if absolute_href:
                        dss_files.append(absolute_href)

    else:
        msg = (
            "Provided path must be either a directory containing "
            "DSS files or a STAC catalog JSON file."
        )
        raise ValueError(msg)

    if not dss_files:
        msg = f"No DSS files found at: {input_path}"
        raise ValueError(msg)

    print(f"✅ Found {len(dss_files)} DSS file(s).")

    return dss_files

In [ ]:
# From folder
files = collect_dss_file_paths("./data/0_source/aorc/indian-creek/indian-creek-aorc-catalog/catalog.json")
files
# From STAC catalog
files = collect_dss_file_paths("./data/0_source/aorc/dss_files")
files


✅ Found 4 DSS file(s).
✅ Found 4 DSS file(s).


['/workspaces/Importance-Sampling-for-SST/data/0_source/aorc/dss_files/20240106.dss',
 '/workspaces/Importance-Sampling-for-SST/data/0_source/aorc/dss_files/20240111.dss',
 '/workspaces/Importance-Sampling-for-SST/data/0_source/aorc/dss_files/20240116.dss',
 '/workspaces/Importance-Sampling-for-SST/data/0_source/aorc/dss_files/20240121.dss']

In [ ]:
import yaml
from pathlib import Path

In [ ]:
project_congig_path = Path("./example-input-data/project_config.yaml").resolve()
with open(project_congig_path, "r") as f:
    config = yaml.safe_load(f)

In [ ]:
config["project"]["name"]

'merced_sst_run'

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from typing import List
from hecdss import HecDss
import contextlib


# ------------------------------------------------------------
# Silence DSS console noise
# ------------------------------------------------------------
@contextlib.contextmanager
def _suppress_stdout_stderr():
    with open(os.devnull, "w") as devnull:
        old_stdout = os.dup(1)
        old_stderr = os.dup(2)
        os.dup2(devnull.fileno(), 1)
        os.dup2(devnull.fileno(), 2)
        try:
            yield
        finally:
            os.dup2(old_stdout, 1)
            os.dup2(old_stderr, 2)


# ------------------------------------------------------------
# Core reader for ONE DSS file
# ------------------------------------------------------------
def _read_dss_cumulative(dss_path: str) -> xr.DataArray:
    """
    Reads all SHG precipitation grids in a DSS file,
    sums them over time, and returns cumulative precipitation
    as a 2D DataArray (y, x).
    """
    dss = HecDss(dss_path)
    catalog = dss.get_catalog()

    # Filter precipitation SHG grids
    paths = [
        p for p in catalog.uncondensed_paths
        if "PRECIPITATION" in p and "SHG" in p
    ]

    if len(paths) == 0:
        dss.close()
        raise ValueError(f"No PRECIPITATION/SHG grids found in {dss_path}")

    data_list = []
    cell_size = None
    ll_x = None
    ll_y = None

    for path in paths:
        record = dss.get(path)
        grid = record.data.astype(np.float32)

        # Replace DSS missing value
        grid[grid == -3.4028235e38] = np.nan
        data_list.append(grid)

        if cell_size is None:
            cell_size = record.cellSize
            ll_x = record.lowerLeftCellX * cell_size
            ll_y = record.lowerLeftCellY * cell_size

    dss.close()

    data_stack = np.stack(data_list, axis=0)
    cumulative = np.nansum(data_stack, axis=0)

    rows, cols = cumulative.shape

    x_coords = ll_x + (0.5 + np.arange(cols)) * cell_size
    y_coords = ll_y + (0.5 + np.arange(rows)) * cell_size

    return xr.DataArray(
        cumulative,
        dims=("y", "x"),
        coords={"x": x_coords, "y": y_coords},
        attrs={"units": "mm", "cell_size": float(cell_size)},
    )


# ------------------------------------------------------------
# Main function you requested
# ------------------------------------------------------------
def process_dss_list(
    dss_files: List[str],
    output_nc: str,
    output_csv: str
):
    """
    Parameters
    ----------
    dss_files : list of str
        List of DSS file paths.
    output_nc : str
        Path to output NetCDF containing cumulative grids.
    output_csv : str
        Path to output CSV containing storm centers.

    Returns
    -------
    None
        Writes:
            - NetCDF file of cumulative grids
            - CSV of storm centers
    """

    cumulative_dict = {}
    storm_centers = []

    for dss_file in dss_files:
        event_name = Path(dss_file).stem

        print(f"Processing {event_name}")

        with _suppress_stdout_stderr():
            da = _read_dss_cumulative(dss_file)

        cumulative_dict[event_name] = da

        # Find max precipitation cell
        arr = da.values

        if np.isnan(arr).all():
            i, j = 0, 0
        else:
            flat = np.nanargmax(arr)
            i, j = np.unravel_index(flat, arr.shape)

        x_center = float(da["x"].values[j])
        y_center = float(da["y"].values[i])

        storm_centers.append({
            "event": event_name,
            "x": x_center,
            "y": y_center
        })

    # --------------------------------------------------------
    # Save cumulative grids as one stacked NetCDF
    # --------------------------------------------------------
    event_names = list(cumulative_dict.keys())

    da_stack = xr.concat(
        [cumulative_dict[name] for name in event_names],
        dim=xr.DataArray(event_names, dims="event", name="event")
    )

    da_stack.name = "cumulative_precip"
    da_stack.to_netcdf(output_nc)

    # --------------------------------------------------------
    # Save storm centers CSV
    # --------------------------------------------------------
    df_centers = pd.DataFrame(storm_centers)
    df_centers.to_csv(output_csv, index=False)

    print(f"\nSaved cumulative grids to: {output_nc}")
    print(f"Saved storm centers to: {output_csv}")

In [ ]:
from glob import glob

process_dss_list(
    dss_files,
    output_nc="./data/1_interim/all_cumulative_precip.nc",
    output_csv="./data/1_interim/storm_centers.csv"
)

Processing 20240106
Processing 20240111
Processing 20240116
Processing 20240121

Saved cumulative grids to: ./data/1_interim/all_cumulative_precip.nc
Saved storm centers to: ./data/1_interim/storm_centers.csv


In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
from pyproj import CRS


# --------------------- CRS (SHG) ---------------------
SHG_WKT = (
    'PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",'
    'GEOGCS["GCS_North_American_1983",'
    'DATUM["D_North_American_1983",'
    'SPHEROID["GRS_1980",6378137.0,298.257222101]],'
    'PRIMEM["Greenwich",0.0],'
    'UNIT["Degree",0.0174532925199433]],'
    'PROJECTION["Albers"],'
    'PARAMETER["False_Easting",0.0],'
    'PARAMETER["Central_Meridian",-96.0],'
    'PARAMETER["Standard_Parallel_1",29.5],'
    'PARAMETER["Standard_Parallel_2",45.5],'
    'PARAMETER["Latitude_Of_Origin",23.0],'
    'UNIT["Meter",1.0]]'
)


def convert_to_shg_and_compute_centers(
    watershed_geojson: str,
    domain_geojson: str,
    output_folder: str
):
    """
    Reads watershed and domain GeoJSONs,
    converts both to SHG CRS,
    saves them as GPKG,
    computes centroids in SHG,
    and saves storm center CSV.

    Outputs:
        - watershed_projected.gpkg
        - domain_projected.gpkg
        - geometry_centers.csv
    """

    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    shg_crs = CRS.from_wkt(SHG_WKT)

    # --------------------------------------------------
    # Read GeoJSONs
    # --------------------------------------------------
    watershed_gdf = gpd.read_file(watershed_geojson)
    domain_gdf = gpd.read_file(domain_geojson)

    if watershed_gdf.crs is None:
        raise ValueError("Watershed GeoJSON has no CRS defined.")
    if domain_gdf.crs is None:
        raise ValueError("Domain GeoJSON has no CRS defined.")

    # --------------------------------------------------
    # Convert to SHG
    # --------------------------------------------------
    watershed_shg = watershed_gdf.to_crs(shg_crs)
    domain_shg = domain_gdf.to_crs(shg_crs)

    # --------------------------------------------------
    # Save projected GPKGs
    # --------------------------------------------------
    watershed_out = output_folder / "watershed_projected.gpkg"
    domain_out = output_folder / "domain_projected.gpkg"

    watershed_shg.to_file(watershed_out, driver="GPKG")
    domain_shg.to_file(domain_out, driver="GPKG")

    # --------------------------------------------------
    # Merge geometries (if multi-feature)
    # --------------------------------------------------
    watershed_geom = watershed_shg.geometry.unary_union
    domain_geom = domain_shg.geometry.unary_union

    # --------------------------------------------------
    # Compute centroids (in meters, SHG projection)
    # --------------------------------------------------
    watershed_centroid = watershed_geom.centroid
    domain_centroid = domain_geom.centroid

    # --------------------------------------------------
    # Save storm center CSV
    # --------------------------------------------------
    centers_df = pd.DataFrame([
        {
            "name": "watershed",
            "x": float(watershed_centroid.x),
            "y": float(watershed_centroid.y)
        },
        {
            "name": "domain",
            "x": float(domain_centroid.x),
            "y": float(domain_centroid.y)
        }
    ])

    centers_out = output_folder / "geometry_centers.csv"
    centers_df.to_csv(centers_out, index=False)

    print(f"Saved watershed GPKG: {watershed_out}")
    print(f"Saved domain GPKG: {domain_out}")
    print(f"Saved centers CSV: {centers_out}")

    return centers_df

In [ ]:
convert_to_shg_and_compute_centers(
    watershed_geojson="./data/0_source/trinity.geojson",
    domain_geojson="./data/0_source/trinity-transpo-area-v01.geojson",
    output_folder="./data/1_interim"
)

Saved watershed GPKG: data/1_interim/watershed_projected.gpkg
Saved domain GPKG: data/1_interim/domain_projected.gpkg
Saved centers CSV: data/1_interim/geometry_centers.csv


/tmp/ipykernel_68038/567980253.py:77: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  watershed_geom = watershed_shg.geometry.unary_union
/tmp/ipykernel_68038/567980253.py:78: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  domain_geom = domain_shg.geometry.unary_union


,name,x,y
0,watershed,-47423.503597,1.019272e+06
1,domain,445653.927772,1.016893e+06


In [2]:
from SSTImportanceSampling import Preprocessor

In [7]:
import os
os.environ["PROJ_LIB"] = "/opt/conda/envs/mbi-base/share/proj"

In [8]:
p = Preprocessor(config_path="./example_config.yaml")

In [9]:
p.run()


 SST Importance Sampling - Preprocessor 

Running preprocessing...


Processing DSS files...

  → 20240106
  → 20240111
  → 20240116
  → 20240121

✅ Preprocessing saved to: /workspaces/Importance-Sampling-for-SST/data/1_interim/indian_creek/indian_creek_sst_ais_20260416_205349_seed8341/preprocessing


✅ Using preprocessing from:
/workspaces/Importance-Sampling-for-SST/data/1_interim/indian_creek/indian_creek_sst_ais_20260416_205349_seed8341/preprocessing
✅ Current run folder:
/workspaces/Importance-Sampling-for-SST/data/1_interim/indian_creek/indian_creek_sst_ais_20260416_205349_seed8341



PosixPath('/workspaces/Importance-Sampling-for-SST/data/1_interim/indian_creek/indian_creek_sst_ais_20260416_205349_seed8341/preprocessing')